In [2]:
from pathlib import Path

import torch 
import pandas as pd

In [3]:
data_path = Path("tinystories_100mb.jsonl")
if not data_path.exists():
    data_path = Path("..") / data_path

all_data = pd.read_json(data_path, lines=True)

In [ ]:
text = all_data["text"].dropna().astype(str).str.cat(sep="\n") + "\n"

In [5]:
print(text[0:100]), print(len(text))

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with
101855429


(None, None)

In [6]:
# text = all_data['text'][1]
print(f"First ten words of text: {text[:10]}\n")
print(f"Length of the text in characters: {len(text)}")
tokens = text.encode('utf-8')
tokens = list(map(int, tokens))
print(f"Length of the text in tokens: {len(tokens)}")
tokens[:10]

First ten words of text: One day, a

Length of the text in characters: 101855429
Length of the text in tokens: 102002112


[79, 110, 101, 32, 100, 97, 121, 44, 32, 97]

In [7]:
def get_pairs(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts
stats = get_pairs(tokens)

# print(sorted(((v, k) for k, v in stats.items()), reverse=True))

In [8]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


def train_bpe(text, vocab_size=1000):
    if vocab_size < 256:
        raise ValueError("Byte-level vocabulary size must be at least 256")

    # UTF-8 bytes become integers from 0 to 255.
    ids = list(text.encode("utf-8"))
    merges = {}

    num_merges = vocab_size - 256

    for i in range(num_merges):
        stats = get_stats(ids)

        if not stats:
            break

        pair = max(stats, key=stats.get)

        # Stop if no pair occurs more than once.
        if stats[pair] < 2:
            break

        new_id = 256 + i
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id

        print(
            f"Merge {i + 1}: {pair} -> {new_id} "
            f"(frequency: {stats[pair]})"
        )

    return ids, merges

In [9]:
ids, merges = train_bpe(
    text[:900000],
    vocab_size=1000,
)

Merge 1: (101, 32) -> 256 (frequency: 32509)
Merge 2: (100, 32) -> 257 (frequency: 23315)
Merge 3: (116, 104) -> 258 (frequency: 17216)
Merge 4: (32, 97) -> 259 (frequency: 13770)
Merge 5: (46, 32) -> 260 (frequency: 13144)
Merge 6: (116, 32) -> 261 (frequency: 12296)
Merge 7: (115, 32) -> 262 (frequency: 10402)
Merge 8: (121, 32) -> 263 (frequency: 10201)
Merge 9: (101, 114) -> 264 (frequency: 10037)
Merge 10: (110, 257) -> 265 (frequency: 9857)
Merge 11: (111, 32) -> 266 (frequency: 8583)
Merge 12: (101, 257) -> 267 (frequency: 8449)
Merge 13: (258, 256) -> 268 (frequency: 8018)
Merge 14: (105, 110) -> 269 (frequency: 7824)
Merge 15: (119, 97) -> 270 (frequency: 7288)
Merge 16: (104, 256) -> 271 (frequency: 7147)
Merge 17: (44, 32) -> 272 (frequency: 7071)
Merge 18: (111, 117) -> 273 (frequency: 6469)
Merge 19: (259, 265) -> 274 (frequency: 6075)
Merge 20: (116, 266) -> 275 (frequency: 5788)
Merge 21: (101, 110) -> 276 (frequency: 5503)
Merge 22: (97, 114) -> 277 (frequency: 4522)
Me

In [10]:
merges

{(101, 32): 256,
 (100, 32): 257,
 (116, 104): 258,
 (32, 97): 259,
 (46, 32): 260,
 (116, 32): 261,
 (115, 32): 262,
 (121, 32): 263,
 (101, 114): 264,
 (110, 257): 265,
 (111, 32): 266,
 (101, 257): 267,
 (258, 256): 268,
 (105, 110): 269,
 (119, 97): 270,
 (104, 256): 271,
 (44, 32): 272,
 (111, 117): 273,
 (259, 265): 274,
 (116, 266): 275,
 (101, 110): 276,
 (97, 114): 277,
 (10, 10): 278,
 (111, 110): 279,
 (111, 109): 280,
 (104, 97): 281,
 (269, 103): 282,
 (32, 115): 283,
 (111, 114): 284,
 (105, 108): 285,
 (32, 268): 286,
 (97, 110): 287,
 (104, 101): 288,
 (259, 32): 289,
 (270, 262): 290,
 (260, 84): 291,
 (264, 32): 292,
 (105, 116): 293,
 (115, 97): 294,
 (108, 108): 295,
 (105, 109): 296,
 (105, 100): 297,
 (114, 101): 298,
 (46, 278): 299,
 (260, 83): 300,
 (105, 262): 301,
 (258, 101): 302,
 (97, 265): 303,
 (115, 116): 304,
 (260, 72): 305,
 (108, 111): 306,
 (105, 114): 307,
 (108, 97): 308,
 (32, 104): 309,
 (305, 256): 310,
 (300, 271): 311,
 (99, 107): 312,
 (116

In [11]:
print(f"Learned merges: {len(merges)}")
print(f"Final token count: {len(ids)}")
print(f"Compression ratio: {len(text[:900000]) / len(ids):.2f} characters per token")
print(f"Length of the text in tokens: {len(text)}")

Learned merges: 744
Final token count: 283331
Compression ratio: 3.18 characters per token
Length of the text in tokens: 101855429


In [12]:
# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [ ]:
vocab = build_vocab(merges)
test_data = text[900000:990000]
encoded_test_data = encode(test_data, merges)
decoded_test_data = decode(encoded_test_data, vocab)

In [20]:
test_data[:100]

'y with his dog. Lily and Ben hugged each other. They said they were happy they were friends. They sa'

In [19]:
decoded_test_data[:100]

'y with his dog. Lily and Ben hugged each other. They said they were happy they were friends. They sa'

- Our tokenizer is working fine

In [23]:
import json
from pathlib import Path


def save_merges(
    merges,
    path="tokenizer/tokenizer.json",
):
    path = Path(path)

    # Create the tokenizer directory if it does not exist.
    path.parent.mkdir(parents=True, exist_ok=True)

    data = {
        "base_vocab_size": 256,
        "vocab_size": 256 + len(merges),
        "merges": [
            [pair[0], pair[1], new_id]
            for pair, new_id in merges.items()
        ],
    }

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)

    print(f"Tokenizer saved to: {path.resolve()}")


save_merges(merges)

Tokenizer saved to: C:\Users\Tvari\Desktop\TvaritRepo\SLM\slm-from-scratch\tokenizer\tokenizer\tokenizer.json
